In [2]:
# [FINAL INTEGRATED SCRIPT FOR THE BIG RUN]
# Target: 25 documents x 36 models = 900 cases
# Standard: Lab Coding Convention (English comments, Type hinting)

import json
import re
import html
import pandas as pd
from kiwipiepy import Kiwi
from enum import Enum
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field


# ==========================================
# 1. Status Definition & Dataclasses
# ==========================================
class Verdict(Enum):
    PASS = "PASS"
    WARNING = "WARNING"
    FAIL = "FAIL"
    ERROR = "ERROR"

@dataclass
class EvalResult:
    verdict: Verdict
    reason: str
    ref_ratio: float
    model_ratio: float
    offender_sentences: List[str] = field(default_factory=list)

# ==========================================
# 2. Micro Engine (Morphological Analysis)
# ==========================================
class SentenceAnalyzer:
    def __init__(self, kiwi_instance: Kiwi) -> None:
        self.kiwi = kiwi_instance
        self.rules: Dict[str, List[str]] = {
            'honorific': ['요', '습니다', 'ㅂ니다', '까', '십시오', '십시다', 'ㅂ시다', '세요', '셔요', '시죠', '대요', '네요', '군요'], 
            'plain': ['다', '어', '아', '야', '니', '냐', '자', '라', '대', '지', '군', '구나', '거라', '렴', '려무나'],
            'archaic': ['는가', '게', '세', '구려', '오', '소', '나', '구먼', '감세', '함세'] # Archaic endings for Phase 3
        }

    def process(self, sentence: str) -> Dict[str, Any]:
        """
        Analyze a single sentence to determine its honorific level and detect specific linguistic constraints.
        """
        # Unescape HTML entities and mask quotes/brackets to ignore direct quotes
        unescaped_sent = html.unescape(sentence)
        masked_sent = re.sub(r'["\'“”‘’\[\]\(\)].*?["\'“”‘’\[\]\(\)]', '', unescaped_sent)
        tokens = self.kiwi.tokenize(masked_sent)
        
        levels = set()
        has_1st_person = any(t.form in ['나', '저', '내', '제', '우리', '저희'] and t.tag == 'NP' for t in tokens)
        has_shi = any(t.form in ['시', '으시'] and t.tag == 'EP' for t in tokens) # Check for subject honorific marker
        has_propositive = any(t.form in ['우리', '같이', '함께'] for t in tokens)

        for i, t in enumerate(tokens):
            if t.tag == 'EF':
                # Ignore indirect quotations
                if any(next_t.tag == 'JKQ' for next_t in tokens[i+1:i+3]): continue
                f = t.form
                
                # Propositive + Imperative mismatch constraint
                if has_propositive and f.endswith('시오'): return {"level": "ERROR", "has_shi": has_shi}
                
                if any(f.endswith(k) for k in self.rules['honorific']): levels.add('HONORIFIC')
                elif any(f == k for k in self.rules['plain']): levels.add('PLAIN')
                elif any(f == k or f.endswith(k) for k in self.rules['archaic']):
                    prev_t = tokens[i-1] if i > 0 else None
                    if prev_t and prev_t.tag == 'VV' and f in ['오', '요']: levels.add('HONORIFIC')
                    else: levels.add('ARCHAIC_WARNING')
                        
        # 1st person pronoun + subject honorific mismatch constraint
        if has_1st_person and has_shi: return {"level": "ERROR", "has_shi": has_shi}

        if 'PLAIN' in levels: return {"level": "PLAIN", "has_shi": has_shi}
        if 'HONORIFIC' in levels: return {"level": "HONORIFIC", "has_shi": has_shi}
        if 'ARCHAIC_WARNING' in levels: return {"level": "ARCHAIC_WARNING", "has_shi": has_shi}
        return {"level": "UNKNOWN", "has_shi": has_shi}

# ==========================================
# 3. Phase 3: HITL (Human-in-the-Loop) Manager
# ==========================================
class HITLManager:
    def __init__(self, transition_threshold: float = 0.5) -> None:
        self.transition_threshold = transition_threshold
        self.queue: List[Dict[str, Any]] = []

    def check_distraction(self, sentences_info: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Calculate the transition rate between marked (-시-) and unmarked honorifics."""
        honorifics = [s for s in sentences_info if s['level'] == 'HONORIFIC']
        if len(honorifics) <= 1: return {"rate": 0.0, "is_distracting": False}

        transitions = sum(1 for i in range(1, len(honorifics)) if honorifics[i-1]['has_shi'] != honorifics[i]['has_shi'])
        rate = transitions / (len(honorifics) - 1)
        return {"rate": rate, "is_distracting": rate >= self.transition_threshold}

    def escalate(self, doc_id: str, model: str, reason: str, text: str) -> None:
        """Push edge cases to the human review queue."""
        self.queue.append({
            "Doc_ID": doc_id, "Model": model, "Reason": reason, "Text": text, "Human_Review": ""
        })

    def export(self, filename: str = "Phase3_HITL_Task.csv") -> None:
        """Export the queue to a CSV file for human review."""
        if self.queue:
            pd.DataFrame(self.queue).to_csv(filename, index=False, encoding='utf-8-sig')

# ==========================================
# 4. Macro Engine & Master Pipeline
# ==========================================
class MasterHonorificEvaluator:
    def __init__(self, data_dict: Dict[str, Any], tolerance: float = 0.10) -> None:
        self.data = data_dict
        self.kiwi = Kiwi()
        self.micro = SentenceAnalyzer(self.kiwi)
        self.hitl = HITLManager(transition_threshold=0.5)
        self.tolerance = tolerance

    def evaluate_specific_model(self, doc_id: str, model_name: str, ref_name: str = 'Human-Reference') -> Optional[EvalResult]:
        """
        Evaluate a specific model's translation against the human reference using dynamic thresholds.
        """
        if doc_id not in self.data or model_name not in self.data[doc_id]: return None
        
        ref_sents = [s.strip() for s in self.data[doc_id][ref_name].split('\n') if s.strip()]
        model_sents = [s.strip() for s in self.data[doc_id][model_name].split('\n') if s.strip()]
        
        ref_info = [self.micro.process(s) for s in ref_sents]
        model_info = [self.micro.process(s) for s in model_sents]
        
        ref_levels = [info['level'] for info in ref_info]
        model_levels = [info['level'] for info in model_info]
        
        res = None 
        
        # 1. Check for explicit grammatical errors
        if "ERROR" in model_levels:
            err_idx = model_levels.index("ERROR")
            res = EvalResult(Verdict.FAIL, "Grammatical Error", 0, 0, [model_sents[err_idx]])

        # 2. Check for archaic endings
        elif "ARCHAIC_WARNING" in model_levels:
            bad_idx = model_levels.index("ARCHAIC_WARNING")
            res = EvalResult(Verdict.WARNING, "Archaic Expression", 0, 0, [model_sents[bad_idx]])
            
        # 3. Check for stylistic distraction (Transition rate)
        else:
            distraction = self.hitl.check_distraction(model_info)
            if distraction['is_distracting']:
                res = EvalResult(Verdict.WARNING, f"Stylistic Distraction ({distraction['rate']*100:.0f}%)", 0, 0, [])
            
            # 4. Dynamic threshold evaluation
            else:
                ref_ratio = ref_levels.count('PLAIN') / max(len(ref_levels), 1)
                mod_ratio = model_levels.count('PLAIN') / max(len(model_levels), 1)
                
                if (ref_ratio - self.tolerance) <= mod_ratio <= (ref_ratio + self.tolerance):
                    res = EvalResult(Verdict.PASS, "Safe Zone", ref_ratio, mod_ratio)
                else:
                    res = EvalResult(Verdict.WARNING, "Style Deviation", ref_ratio, mod_ratio)

        # Escalate non-PASS results to HITL queue
        if res.verdict != Verdict.PASS:
            sample_text = res.offender_sentences[0] if res.offender_sentences else model_sents[0]
            self.hitl.escalate(doc_id, model_name, res.reason, sample_text)
            
        return res

if __name__ == "__main__":
    # 1. Data Loading & Joining (WMT25 Reference + Model Outputs)
    # Ensure your paths are correct for the lab server environment
    wmt_refs_path = '/home/yoonhwa-song/workspace/ape_inference/data/wmt25/wmt25-genmt.jsonl'
    models_path = '/home/yoonhwa-song/workspace/ape_inference/data/wmt25/tgt_doc.json'
    
    data: Dict[str, Any] = {}
    korean_refs: List[str] = []
    
    try:
        # Filter WMT25 for Korean General Domain
        with open(wmt_refs_path, 'r', encoding='utf-8') as f:
            for line in f:
                row = json.loads(line.strip())
                if (row.get('tgt_lang', '').startswith('ko') and 
                    row.get('collection_id') == "general" and 
                    row.get('domain') in ["literary", "social", "news"]):
                    korean_refs.append(row['refs']['refA']['ref'])
                    
        with open(models_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # Map references to model outputs by sequence (The 'Zip' strategy)
        for i, doc_id in enumerate(data.keys()):
            if i < len(korean_refs):
                data[doc_id]['Human-Reference'] = korean_refs[i]
                
        print(f"✅ Data Ready: {len(data)} documents mapped with human references.")
        
    except Exception as e:
        print(f"🚨 Data Loading Error: {e}")

    # 2. Initialize Master Evaluator
    evaluator = MasterHonorificEvaluator(data_dict=data, tolerance=0.10)
    all_models = [m for m in data['0'].keys() if m != 'Human-Reference']
    
    # 3. THE BIG RUN: 25 x 36 loop
    print("=" * 80)
    print(f"🔥 Starting The Big Run: {len(data)} docs x {len(all_models)} models")
    print("=" * 80)
    
    grand_stats = {'PASS': 0, 'WARNING': 0, 'FAIL': 0}
    
    for doc_id in data.keys():
        if 'Human-Reference' not in data[doc_id]: continue
        
        for model in all_models:
            res = evaluator.evaluate_specific_model(doc_id=doc_id, model_name=model)
            if res:
                grand_stats[res.verdict.name] += 1
                
    print("=" * 80)
    print(f"🏆 Final Statistics: PASS {grand_stats['PASS']} | WARNING {grand_stats['WARNING']} | FAIL {grand_stats['FAIL']}")
    
    # 4. Export Final Audit Log for Human-in-the-Loop
    evaluator.hitl.export("Final_900_Honorific_Audit_Log.csv")
    print("✅ Full results exported to 'Final_900_Honorific_Audit_Log.csv'.")

✅ Data Ready: 25 documents mapped with human references.
🔥 Starting The Big Run: 25 docs x 36 models
🏆 Final Statistics: PASS 330 | WARNING 446 | FAIL 124
✅ Full results exported to 'Final_900_Honorific_Audit_Log.csv'.
